# Notebook 04 - ECG Autoencoder AD8232

Train Conv1D Autoencoder tren nhip ECG binh thuong tu MIT-BIH de tao model phat hien bat thuong ECG. Dau vao model la cua so 250 mau, phu hop voi AD8232 dang doc 5 giay @ 50Hz tren ESP32-S3.

In [ ]:
# Cell 1: Cai thu vien tren Google Colab
# Colab da co san TensorFlow. Khong cai lai tensorflow de tranh keo pandas 3.x gay conflict.
# Pin numpy 1.26.4 de tranh loi wfdb: Python integer 256 out of bounds for uint8.
# Neu Colab hien canh bao package khac can numpy>=2 thi co the bo qua cho notebook nay.
# Sau cell nay: Runtime -> Restart session, roi chay lai tu Cell 2.
!pip install -q "numpy==1.26.4" "pandas==2.2.2" "wfdb==4.1.2" scipy

In [ ]:
# Cell 2: Import va cau hinh
from pathlib import Path
import shutil

import numpy as np
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping

RECORDS = ['100', '101', '103', '106', '107']
MITDB_DIR = Path('./mitdb')
WINDOW_SIZE = 250
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)
print('Import OK')

In [ ]:
# Cell 3: Tai MIT-BIH Arrhythmia Database (~200MB, lan dau)
if not MITDB_DIR.exists() or not any(MITDB_DIR.glob('*.dat')):
    wfdb.dl_database('mitdb', str(MITDB_DIR))
else:
    print('MIT-BIH da ton tai, bo qua buoc download')

In [ ]:
# Cell 4: Ham load ECG va tao sliding windows theo skill ecg-sensor
def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=3):
    b, a = butter(order, [lowcut, highcut], btype='band', fs=fs)
    return filtfilt(b, a, signal)


def load_normal_ecg(record_ids=RECORDS, window=WINDOW_SIZE):
    """Chi lay nhip binh thuong symbol='N' de train autoencoder."""
    windows = []
    for record_id in record_ids:
        try:
            record_path = str(MITDB_DIR / record_id)
            record = wfdb.rdrecord(record_path)
            annotation = wfdb.rdann(record_path, 'atr')

            fs = int(record.fs)
            ecg = record.p_signal[:, 0].astype(np.float32)
            ecg = bandpass_filter(ecg, fs)
            ecg = (ecg - ecg.min()) / (ecg.max() - ecg.min() + 1e-8)

            normal_beats = [sample for sample, symbol in zip(annotation.sample, annotation.symbol) if symbol == 'N']
            for beat in normal_beats[5:-5]:
                start = beat - window // 2
                end = start + window
                if start > 0 and end < len(ecg):
                    windows.append(ecg[start:end])

            print(f'Record {record_id}: {len(normal_beats)} nhip binh thuong')
        except Exception as exc:
            print(f'Loi record {record_id}: {exc}')

    arr = np.asarray(windows, dtype=np.float32).reshape(-1, window, 1)
    print(f'Tong: {len(arr)} windows tu {len(record_ids)} records')
    return arr


print('Helpers OK')

In [ ]:
# Cell 5: Load records 100,101,103,106,107 va chi lay nhip binh thuong symbol='N'
X = load_normal_ecg()
print('Shape:', X.shape)
if len(X) == 0:
    raise RuntimeError('Khong tao duoc ECG window nao. Hay Restart session sau khi cai numpy==1.26.4 va chay lai tu dau.')

idx = np.random.choice(len(X), size=min(3, len(X)), replace=False)
plt.figure(figsize=(12, 3))
for i, sample_idx in enumerate(idx, 1):
    plt.subplot(1, len(idx), i)
    plt.plot(X[sample_idx, :, 0])
    plt.ylim(0, 1)
    plt.title(f'Window {sample_idx}')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Build Conv1D Autoencoder (~50KB)
def build_ecg_autoencoder():
    inputs = layers.Input(shape=(WINDOW_SIZE, 1), name='ecg_window')

    x = layers.Conv1D(16, kernel_size=3, padding='same', activation='relu')(inputs)
    x = layers.MaxPooling1D(pool_size=5, padding='same')(x)      # 250 -> 50
    x = layers.Conv1D(8, kernel_size=3, padding='same', activation='relu')(x)
    encoded = layers.MaxPooling1D(pool_size=5, padding='same', name='latent_10x8')(x)  # 50 -> 10

    x = layers.Conv1D(8, kernel_size=3, padding='same', activation='relu')(encoded)
    x = layers.UpSampling1D(size=5)(x)                           # 10 -> 50
    x = layers.Conv1D(16, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.UpSampling1D(size=5)(x)                           # 50 -> 250
    outputs = layers.Conv1D(1, kernel_size=3, padding='same', activation='sigmoid', name='reconstruction')(x)

    model = Model(inputs, outputs, name='ecg_conv1d_autoencoder')
    model.compile(optimizer='adam', loss='mse')
    return model


model = build_ecg_autoencoder()
model.summary()

In [ ]:
# Cell 7: Train 50 epochs, EarlyStopping patience=8
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True
)

history = model.fit(
    X, X,
    validation_split=0.15,
    epochs=50,
    batch_size=64,
    callbacks=[
        early_stop,
        tf.keras.callbacks.ReduceLROnPlateau(patience=4, factor=0.5)
    ],
    verbose=1
)

plt.figure(figsize=(7, 4))
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 8: Tinh reconstruction error va threshold 95 percentile
reconstructions = model.predict(X, batch_size=128, verbose=0)
reconstruction_errors = np.mean(np.abs(reconstructions - X), axis=(1, 2))
threshold = np.percentile(reconstruction_errors, 95)

print(f'Mean reconstruction error: {reconstruction_errors.mean():.6f}')
print(f'P95 threshold: {threshold:.6f}')

plt.figure(figsize=(7, 4))
plt.hist(reconstruction_errors, bins=60)
plt.axvline(threshold, color='r', linestyle='--', label=f'threshold={threshold:.6f}')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Cell 9: Kiem tra reconstruction truc quan
sample = X[:3]
pred = model.predict(sample, verbose=0)

plt.figure(figsize=(12, 6))
for i in range(len(sample)):
    plt.subplot(len(sample), 1, i + 1)
    plt.plot(sample[i, :, 0], label='input')
    plt.plot(pred[i, :, 0], label='reconstruction', alpha=0.8)
    plt.ylim(0, 1)
    plt.legend(loc='upper right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 10: Luu model va threshold
model.save('ecg_autoencoder.h5', include_optimizer=False)
np.save('ecg_threshold.npy', np.asarray(threshold, dtype=np.float32))

print('Saved: ecg_autoencoder.h5')
print('Saved: ecg_threshold.npy')
print('Model size KB:', Path('ecg_autoencoder.h5').stat().st_size / 1024)

In [ ]:
# Cell 11: Download tren Colab va copy vao thu muc models/ neu project da mount trong Colab
try:
    from google.colab import files
    files.download('ecg_autoencoder.h5')
    files.download('ecg_threshold.npy')
except Exception as exc:
    print('Khong phai Colab hoac khong download duoc:', exc)

candidate_model_dirs = [
    Path('/content/aiot_health/models'),
    Path('/content/drive/MyDrive/aiot_health/models'),
    Path('../../models'),
    Path('models'),
]

for model_dir in candidate_model_dirs:
    if model_dir.exists():
        shutil.copy2('ecg_autoencoder.h5', model_dir / 'ecg_autoencoder.h5')
        shutil.copy2('ecg_threshold.npy', model_dir / 'ecg_threshold.npy')
        print(f'Copied artifacts to: {model_dir.resolve()}')
        break
else:
    print('Chua tim thay thu muc models/. Hay copy 2 file tren vao models/ cua project sau khi download.')